In [ ]:
import sys
from pathlib import Path

# Add repo root to Python path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import pandas as pd
from src.data import load_raw_csv
from src.prep import prepare_complaints_df, stratified_cap_per_class, time_split

# Define column names
TEXT_COL = "narrative"
LABEL_COL = "Product"
DATE_COL  = "Date received"

# Load raw data
df = load_raw_csv("../data/raw")
print("Raw shape:", df.shape)

df = prepare_complaints_df(df, text_col=TEXT_COL, label_col=LABEL_COL, date_col=DATE_COL, min_chars=20)
print("After basic cleaning:", df.shape)

df[LABEL_COL].value_counts().head(10)


Product
Credit reporting, credit repair services, or other personal consumer reports    806929
Credit reporting or other personal consumer reports                             366266
Debt collection                                                                 266630
Mortgage                                                                        119106
Credit card or prepaid card                                                     108656
Checking or savings account                                                     100419
Credit card                                                                      50359
Student loan                                                                     44236
Money transfer, virtual currency, or money service                               41495
Vehicle loan or lease                                                            32065
Name: count, dtype: int64

In [ ]:
top_n = 10
top_labels = df[LABEL_COL].value_counts().head(top_n).index
df = df[df[LABEL_COL].isin(top_labels)].copy()

CAP_PER_CLASS = 20000   # try 10k if you want even faster
SEED = 42

df_small = stratified_cap_per_class(df, label_col=LABEL_COL, cap_per_class=CAP_PER_CLASS, random_state=SEED)

print("Sampled shape:", df_small.shape)
print(df_small[LABEL_COL].value_counts())


In [ ]:
# Split data into train and test based on date
df = df.sort_values(DATE_COL)
cut = int(len(df) * 0.8)

train = df.iloc[:cut]
test  = df.iloc[cut:]

X_train, y_train = train["text_clean"], train[LABEL_COL]
X_test,  y_test  = test["text_clean"],  test[LABEL_COL]

len(train), len(test), y_train.nunique()

(1611170, 402793, 15)

In [ ]:
# Baseline: TF-IDF + Logistic Regression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report

tfidf_sgd = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=80000,      # reduce if slow / low RAM
        ngram_range=(1,2),       # you can switch to (1,1) if still slow
        min_df=5
    )),
    ("sgd", SGDClassifier(
        loss="log_loss",         # logistic regression style
        max_iter=25,
        tol=1e-3,
        n_jobs=-1,
        random_state=42
    ))
])

tfidf_sgd.fit(X_train, y_train)
pred = tfidf_sgd.predict(X_test)

macro = f1_score(y_test, pred, average="macro")
weighted = f1_score(y_test, pred, average="weighted")

print("TF-IDF + SGD Macro F1:", macro)
print("TF-IDF + SGD Weighted F1:", weighted)
print(classification_report(y_test, pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Plot confusion matrix for top N labels
labels = list(top_labels)
ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    labels=labels,
    xticks_rotation=90
)
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd

# Save baseline metrics
Path("../reports").mkdir(exist_ok=True)

baseline_metrics = pd.DataFrame([{
    "model": "TF-IDF + SGD(log_loss)",
    "macro_f1": macro,
    "weighted_f1": weighted,
    "top_n_products": top_n,
    "cap_per_class": CAP_PER_CLASS,
    "vectorizer": "TF-IDF (1-2 grams, max_features=80000, min_df=5)",
    "split": "time split 80/20"
}])

baseline_metrics.to_csv("../reports/baseline_metrics.csv", index=False)
baseline_metrics